### Analyse av dag-data

- Les inn data
- Aggreger opp til dagsnivå
- Join med info df
- 

### Imports/ Snowflake session

In [11]:
from snowflake.snowpark.functions import col, sum as sum_, max as max_, datediff, round as round_, year, month, when, lit, lag, avg, count, stddev, to_date, dayofweek, weekofyear, dayofmonth, quarter, last_day
from snowflake.snowpark import Window
import re
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# from dwh_tools.get_or_create_session import get_or_create_session
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session
import os
import requests
import gzip
import json
# from dwh_tools.get_or_create_session import get_or_create_session
pd.set_option('display.max_columns', None)
import time
import pywin

In [12]:
def get_or_create_session(schema: str = None) -> Session:
    """
    Returns the active Snowpark session if running inside Snowflake,
    otherwise creates one locally using Azure OAuth (interactive login if needed).
 
    Parameters
    ----------
    config_path : str
        Path to a JSON config file containing Snowflake connection parameters
        (account, warehouse, role, database, schema).
 
    Returns
    -------
    Session
        A Snowpark Session object.
    """
    # If running on snowflake
    if 'POSIT_PRODUCT' in os.environ:
        session = Session.builder.getOrCreate()
        session.sql("USE DATABASE PROD_FOR_SKADE_PRODUKT_ADHOC").collect()
        if schema:
            session.sql("USE SCHEMA " + schema).collect()
        else:
            session.sql("USE SCHEMA PRODUKT_WRITE_DEV").collect()
        session.sql("USE WAREHOUSE SKADE_VWH").collect()
 
        return session
 
    try:
        session = get_active_session()
        return session
    except Exception:
        import win32api
        if schema is None:
            schema = 'PRODUKT_WRITE_DEV'
 
        connection_parameters = {
            "server": "km28161.west-europe.azure.snowflakecomputing.com",
            "warehouse": "SKADE_VWH",
            "account": "VK82539-KLP",
            "database": "PROD_FOR_SKADE_PRODUKT_ADHOC",
            "schema" : schema,
            "user": win32api.GetUserNameEx(win32api.NameUserPrincipal),  
            "authenticator": "externalbrowser"
        }
       
        # Create the session
        session = Session.builder.configs(connection_parameters).create()
        return session

In [13]:
session = get_or_create_session()

### Les data

In [14]:
# Kundesenter data
df = session.table('elh_write.inngangsdata').to_pandas()
df.columns = df.columns.str.lower()
# TIA data
df_info = session.table('inngangsdata_info').to_pandas()
df_info.columns = df_info.columns.str.lower()

In [15]:
df.head()

,unique_id,forste_kogruppe,forste_ko,kunde,ankomst_dato,tid_i_ko,behandlet,behandlingstid,etterbehandlingstid,behandlingstid_etterbehandlingstid,overfort_eksternt,arbeidsplass,agent_id,god_reserve
0,3051408,SK_Salg_Ekst_PM,SK_Salg_Ekst_PM_Tlf,99525091,02.06.2026 15:57:52,3,Behandlet,01:32,01:29,03:01,None,wp3409,lni,God agent
1,3051415,SK_Salg_Ny_PM,SK_Salg_Ny_PM_Tlf,0046707954838,02.06.2026 15:57:37,16,Behandlet,07:36,00:47,08:23,None,wp3404,edm,God agent
2,3051412,SK_Salg_Ekst_PM,SK_Salg_Ekst_PM_Tlf,97322088,02.06.2026 15:55:05,3,Behandlet,09:58,00:00,09:58,None,wp3403,ba9,God agent
3,3051410,SK_Salg_Ekst_PM,SK_Salg_Ekst_PM_Tlf,92433365,02.06.2026 15:51:43,86,Behandlet,06:22,03:00,09:22,None,wp3342,njn,God agent
4,3051403,SK_Salg_Ekst_PM,SK_Salg_Ekst_PM_Tlf,91521443,02.06.2026 15:51:41,34,Behandlet,03:20,01:30,04:50,None,wp3409,lni,God agent


In [16]:
df_info.head()

,hf_dato,antall_hf_b30_upr01_ny,antall_hf_b7_upr01_ny,antall_hf_f30_upr01_ny,antall_hf_f7_upr01_ny,antall_hf_b30_upr01_for,stddev_premieendring_b30_upr01_for,snitt_premieendring_b30_upr01_for,antall_hf_b7_upr01_for,stddev_premieendring_b7_upr01_for,snitt_premieendring_b7_upr01_for,antall_hf_f30_upr01_for,stddev_premieendring_f30_upr01_for,snitt_premieendring_f30_upr01_for,antall_hf_f7_upr01_for,stddev_premieendring_f7_upr01_for,snitt_premieendring_f7_upr01_for,antall_hf_b30_eph01_ny,antall_hf_b7_eph01_ny,antall_hf_f30_eph01_ny,antall_hf_f7_eph01_ny,antall_hf_b30_eph01_for,stddev_premieendring_b30_eph01_for,snitt_premieendring_b30_eph01_for,antall_hf_b7_eph01_for,stddev_premieendring_b7_eph01_for,snitt_premieendring_b7_eph01_for,antall_hf_f30_eph01_for,stddev_premieendring_f30_eph01_for,snitt_premieendring_f30_eph01_for,antall_hf_f7_eph01_for,stddev_premieendring_f7_eph01_for,snitt_premieendring_f7_eph01_for,antall_hf_b30_epf01_ny,antall_hf_b7_epf01_ny,antall_hf_f30_epf01_ny,antall_hf_f7_epf01_ny,antall_hf_b30_mpb01_for,stddev_premieendring_b30_mpb01_for,snitt_premieendring_b30_mpb01_for,antall_hf_b7_mpb01_for,stddev_premieendring_b7_mpb01_for,snitt_premieendring_b7_mpb01_for,antall_hf_f30_mpb01_for,stddev_premieendring_f30_mpb01_for,snitt_premieendring_f30_mpb01_for,antall_hf_f7_mpb01_for,stddev_premieendring_f7_mpb01_for,snitt_premieendring_f7_mpb01_for,antall_hf_b30_epf01_for,stddev_premieendring_b30_epf01_for,snitt_premieendring_b30_epf01_for,antall_hf_b7_epf01_for,stddev_premieendring_b7_epf01_for,snitt_premieendring_b7_epf01_for,antall_hf_f30_epf01_for,stddev_premieendring_f30_epf01_for,snitt_premieendring_f30_epf01_for,antall_hf_f7_epf01_for,stddev_premieendring_f7_epf01_for,snitt_premieendring_f7_epf01_for,antall_hf_b30_mpb01_ny,antall_hf_b7_mpb01_ny,antall_hf_f30_mpb01_ny,antall_hf_f7_mpb01_ny,aar,kvartal,maaned,ukenummer,ukedag,dag_i_maaned,er_helg,helligdag,er_helligdag,er_dag_foer_helligdag,er_dag_etter_helligdag
0,2025-02-20,374,110,608,127,1933.0,0.094422,1.070897,789.0,0.094622,1.065560,2892.0,0.107997,1.076590,509.0,0.111707,1.085747,690.0,218.0,1023.0,195.0,4045.0,0.129114,1.213016,1631.0,0.122197,1.217828,6001.0,0.129920,1.208837,1082.0,0.117830,1.197224,77.0,28.0,108.0,22.0,5339.0,0.207513,1.080040,2185.0,0.195618,1.077522,7466.0,0.211598,1.083385,1357.0,0.229188,1.083783,851.0,0.106255,1.231999,322.0,0.110300,1.230828,1339.0,0.120008,1.234079,232.0,0.139262,1.232192,1869,558,2789,567,2025,1,2,8,4,20,0,vinterferie,1,0,0
1,2024-09-29,394,100,458,104,2476.0,0.099756,1.029427,428.0,0.078594,1.031192,2713.0,0.079313,1.026227,426.0,0.077383,1.035090,829.0,176.0,896.0,242.0,5295.0,0.150132,1.224554,927.0,0.148647,1.213767,5811.0,0.172913,1.255338,949.0,0.190271,1.255371,96.0,16.0,110.0,30.0,6887.0,0.245302,1.169826,1260.0,0.269208,1.173719,7518.0,0.235157,1.160767,1267.0,0.226781,1.171494,1013.0,0.116281,1.142095,189.0,0.119021,1.159591,1196.0,0.189802,1.246464,194.0,0.242175,1.281829,2135,550,2542,591,2024,3,9,39,0,29,1,None,0,1,0
2,2024-03-26,490,109,454,98,2627.0,0.109980,1.040253,785.0,0.117697,1.042281,2218.0,0.077698,1.038548,310.0,0.070170,1.041273,942.0,212.0,832.0,172.0,5512.0,0.098705,1.095519,1672.0,0.102338,1.089161,4918.0,0.111241,1.103507,736.0,0.110076,1.098018,185.0,44.0,128.0,30.0,6993.0,0.195048,1.146861,2033.0,0.181048,1.151856,6331.0,0.713294,1.151276,905.0,0.217519,1.149786,1245.0,0.055167,1.057240,422.0,0.072891,1.060828,967.0,0.131579,1.066243,155.0,0.092767,1.065313,2641,640,2546,376,2024,1,3,13,2,26,0,None,0,0,0
3,2024-04-29,436,92,495,129,2233.0,0.077661,1.039028,377.0,0.066319,1.042066,2357.0,0.068022,1.019347,388.0,0.078047,1.026614,795.0,149.0,896.0,250.0,4970.0,0.111191,1.104167,851.0,0.098523,1.104260,5010.0,0.120733,1.105260,901.0,0.091335,1.096380,116.0,13.0,166.0,46.0,6416.0,0.708483,1.151797,1121.0,0.196546,1.151688,6580.0,0.214499,1.149186,1105.0,0.266085,1.149236,980.0,0.131053,1.066685,191.0,0.186892,1.056766,985.0,0.126045,1.066029,175.0,0.053088,1.069673,2489,550,2545,630,202

### Databehandling

In [17]:
# Datetime format
df['ankomst_dato'] = pd.to_datetime(df['ankomst_dato'], format="%d.%m.%Y %H:%M:%S").dt.date
# Respons (behandlingstid) som sekunder
df['behandlingstid'] = (
    df['behandlingstid']
    .str.split(':')
    .apply(lambda x: int(x[0]) * 60 + int(x[1]))
)
# Også etterbehandlingstid som sekunder:
df['etterbehandlingstid'] = (
    df['etterbehandlingstid']
    .str.split(':')
    .apply(lambda x: int(x[0]) * 60 + int(x[1]))
)
# Datetime format
df_info['hf_dato'] = pd.to_datetime(df_info['hf_dato'], format="%Y-%m-%d").dt.date

# Behandlet som true/false:
df["behandlet"]=df["behandlet"].eq("Behandlet")
# Lage antall samtaler dag (trenger ikke denne, teller heller unike rader)
# df["antall_samtaler_dag"] = df.groupby("ankomst_dato")["unique_id"].transform("size")

In [18]:
# Få df aggregert opp på dagsnivå før joining med df_info
df = df.copy()

df_dag = (
    df.groupby("ankomst_dato", as_index=False)
    .agg(
        antall_samtaler=("unique_id", "size"),
        behandlet_andel=("behandlet", "mean"),
        tid_i_ko_snitt=("tid_i_ko", "mean"),
        behandlingstid_snitt=("behandlingstid", "mean"),
        total_behandlingstid=("behandlingstid", "sum"),
        etterbehandligstid_snitt=("etterbehandlingstid", "mean"), 
        total_etterbehandligstid=("etterbehandlingstid", "sum")
    ).
    sort_values("ankomst_dato")
    .reset_index(drop=True)
)



In [19]:
df_dag.head()

,ankomst_dato,antall_samtaler,behandlet_andel,tid_i_ko_snitt,behandlingstid_snitt,total_behandlingstid,etterbehandligstid_snitt,total_etterbehandligstid
0,2024-06-03,181,0.850829,274.342541,281.049724,50870,130.419890,23606
1,2024-06-04,408,0.828431,732.632353,280.009804,114244,121.987745,49771
2,2024-06-05,294,0.782313,650.248299,258.166667,75901,97.272109,28598
3,2024-06-06,313,0.808307,447.830671,294.220447,92091,91.447284,28623
4,2024-06-07,371,0.867925,791.533693,301.137466,111722,133.552561,49548


### Joine df og df_info på dag (aggregert)

In [20]:
df_ = pd.merge(df_dag, df_info, left_on="ankomst_dato", right_on="hf_dato", how="left")
df_.head()

,ankomst_dato,antall_samtaler,behandlet_andel,tid_i_ko_snitt,behandlingstid_snitt,total_behandlingstid,etterbehandligstid_snitt,total_etterbehandligstid,hf_dato,antall_hf_b30_upr01_ny,antall_hf_b7_upr01_ny,antall_hf_f30_upr01_ny,antall_hf_f7_upr01_ny,antall_hf_b30_upr01_for,stddev_premieendring_b30_upr01_for,snitt_premieendring_b30_upr01_for,antall_hf_b7_upr01_for,stddev_premieendring_b7_upr01_for,snitt_premieendring_b7_upr01_for,antall_hf_f30_upr01_for,stddev_premieendring_f30_upr01_for,snitt_premieendring_f30_upr01_for,antall_hf_f7_upr01_for,stddev_premieendring_f7_upr01_for,snitt_premieendring_f7_upr01_for,antall_hf_b30_eph01_ny,antall_hf_b7_eph01_ny,antall_hf_f30_eph01_ny,antall_hf_f7_eph01_ny,antall_hf_b30_eph01_for,stddev_premieendring_b30_eph01_for,snitt_premieendring_b30_eph01_for,antall_hf_b7_eph01_for,stddev_premieendring_b7_eph01_for,snitt_premieendring_b7_eph01_for,antall_hf_f30_eph01_for,stddev_premieendring_f30_eph01_for,snitt_premieendring_f30_eph01_for,antall_hf_f7_eph01_for,stddev_premieendring_f7_eph01_for,snitt_premieendring_f7_eph01_for,antall_hf_b30_epf01_ny,antall_hf_b7_epf01_ny,antall_hf_f30_epf01_ny,antall_hf_f7_epf01_ny,antall_hf_b30_mpb01_for,stddev_premieendring_b30_mpb01_for,snitt_premieendring_b30_mpb01_for,antall_hf_b7_mpb01_for,stddev_premieendring_b7_mpb01_for,snitt_premieendring_b7_mpb01_for,antall_hf_f30_mpb01_for,stddev_premieendring_f30_mpb01_for,snitt_premieendring_f30_mpb01_for,antall_hf_f7_mpb01_for,stddev_premieendring_f7_mpb01_for,snitt_premieendring_f7_mpb01_for,antall_hf_b30_epf01_for,stddev_premieendring_b30_epf01_for,snitt_premieendring_b30_epf01_for,antall_hf_b7_epf01_for,stddev_premieendring_b7_epf01_for,snitt_premieendring_b7_epf01_for,antall_hf_f30_epf01_for,stddev_premieendring_f30_epf01_for,snitt_premieendring_f30_epf01_for,antall_hf_f7_epf01_for,stddev_premieendring_f7_epf01_for,snitt_premieendring_f7_epf01_for,antall_hf_b30_mpb01_ny,antall_hf_b7_mpb01_ny,antall_hf_f30_mpb01_ny,antall_hf_f7_mpb01_ny,aar,kvartal,maaned,ukenummer,ukedag,dag_i_maaned,er_helg,helligdag,er_helligdag,er_dag_foer_helligdag,er_dag_etter_helligdag
0,2024-06-03,181,0.850829,274.342541,281.049724,50870,130.419890,23606,2024-06-03,505,133,599,97,2338.0,0.067888,1.019763,387.0,0.073937,1.028876,2659.0,0.101024,1.029085,327.0,0.102971,1.028102,865.0,180.0,786.0,192.0,4921.0,0.125600,1.109447,848.0,0.130024,1.121494,5421.0,0.137216,1.148447,637.0,0.127443,1.143180,163.0,36.0,114.0,21.0,6558.0,0.205987,1.150124,1165.0,0.235064,1.156063,7057.0,0.209951,1.164113,906.0,0.247436,1.157046,988.0,0.145015,1.069494,193.0,0.167753,1.082447,1156.0,0.098263,1.077020,146.0,0.088862,1.066647,2399,498,2291,482,2024,2,6,23,1,3,0,None,0,0,0
1,2024-06-04,408,0.828431,732.632353,280.009804,114244,121.987745,49771,2024-06-04,499,135,607,104,2343.0,0.068691,1.020142,386.0,0.079320,1.031071,2653.0,0.099018,1.028158,311.0,0.087818,1.023989,847.0,188.0,776.0,181.0,4949.0,0.129050,1.111346,876.0,0.137985,1.125973,5431.0,0.137472,1.148206,652.0,0.132029,1.143005,155.0,34.0,116.0,19.0,6586.0,0.207036,1.150978,1156.0,0.233963,1.162051,7064.0,0.209988,1.164319,908.0,0.244356,1.156959,996.0,0.144577,1.069357,196.0,0.166703,1.078869,1139.0,0.098268,1.078314,132.0,0.087905,1.070917,2405,509,2327,499,2024,2,6,23,2,4,0,None,0,0,0
2,2024-06-05,294,0.782313,650.248299,258.166667,75901,97.272109,28598,2024-06-05,502,136,620,119,2356.0,0.072024,1.020849,394.0,0.094595,1.035051,2647.0,0.099156,1.028247,511.0,0.082718,1.023374,835.0,200.0,790.0,175.0,4947.0,0.129070,1.111281,852.0,0.138511,1.128818,5415.0,0.137465,1.147596,1041.0,0.134719,1.147604,148.0,34.0,120.0,19.0,6599.0,0.206497,1.151824,1125.0,0.224133,1.167557,7021.0,0.209893,1.164730,1412.0,0.221535,1.158163,1023.0,0.143443,1.069346,214.0,0.162592,1.078473,1139.0,0.097879,1.078668,201.0,0.125821,1.072454,2392,508,2362,516,2024,2,6,23,3,5,0,None,0,0,0
3,2024-06-06,313,0.808307,447.830671,294.220447,92091,91.447284,28623,2024-06-06,489,123,620,129,2371.0,0.072046,1.021069,402.0,0.095643,1.